In [1]:
!pip install -qq adapters bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.2/302.2 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 130.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 87.0 MB/s eta 0:00:00


In [2]:
import os, torch
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # stack synchrone
torch.cuda.empty_cache()

In [3]:
%%capture
!pip install evaluate jiwer tensorboard datasets[audio] librosa soundfile

In [4]:
import json
from datasets import Dataset, Audio
import torch

In [5]:
import torchaudio

In [6]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    try:
        HF_TOKEN = UserSecretsClient().get_secret("HUGGING_FACE_TOKEN")
    except Exception as exc:
        raise ValueError("HF_TOKEN is required via environment or Kaggle secret HUGGING_FACE_TOKEN") from exc

login(token=HF_TOKEN)

In [8]:
path = "/kaggle/input/fr-audio-dataset/Collecte_Audio_FR_Radiologie-20260119T004052Z-3-001/Collecte_Audio_FR_Radiologie"

In [9]:
# Charger le dictionnaire dico_corpus.json
with open(path + '/dico_corpus.json', 'r') as f:
  dico_corpus = json.load(f)

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/fr-audio-dataset/Collecte_Audio_FR_Radiologie-20260119T004052Z-3-001/Collecte_Audio_FR_Radiologie/dico_corpus.json'

### Construction d’une liste d’enregistrements

In [ ]:
records = []
for idx, meta in dico_corpus.items():
    try:
        folder = f"{path}/{meta['identifiant']}"
        # parcourir tous les fichiers .wav (ou .mp3) dans le dossier
        for fname in os.listdir(folder):
            if fname.endswith(('.wav', '.flac', '.mp3', '.webm')):
                records.append({
                    "audio": os.path.join(folder, fname),
                    "text": meta['contenu']
                })
    except FileNotFoundError:
        print(f"Dossier introuvable ou vide : {folder}")
    except Exception as e:
        print(f"Erreur inattendue pour {folder} : {e}")

In [ ]:
# print(records)

#### Normalisation de texte

In [ ]:
def normalize_text(text):
    text = text.lower()
    text = text.replace('\n', ' ')
    # text = re.sub(r'[^\w\s]', ' ', text)  # Remplacer ponctuation par espace
    # text = re.sub(r'(\d+)', r' \1 ', text)  # Espacer les chiff
    return text

# records = list(map(lambda r: normalize_text(r["text"]), records))
for record in records:
    record['text'] = normalize_text(record['text'])

In [ ]:
print(records[0])

#### Reservation d'ensemble de test après entraînement

In [ ]:
eval_audios = [
    # === 17 AUDIOS DÉJÀ ISOLÉS ===
    "/report_NxSmAyYz42CeH1IRNJXaSVSHMRR_WJ0IF0bvxM7z2Ak/recorded_audio.wav",
    "/report_oqzb78c_pgfdiw1j5VE7h2EbK8pDy6JzqpeZG0SBT2Q/recorded_audio.wav",
    "/report_Kl4hVMi1xNQayscnSkVzc3xVrE7s88OdydZqFsf5CMM/recorded_audio.wav",
    "/report_aOw5Iu1dR7gITUx4qsYWzVqkf8z1FaHIvXKc4vZS2LA/recorded_audio.wav",
    "/report_KXz0PIoW4XbD-b2l7CESbvRebk3l57NMa5-hcUrl6Co/recorded_audio.wav",
    "/report_2TnSIhg-drDPZ2eu54sk6Rj36XSSUxm_NqpZKI7BYyg/recorded_audio.wav",
    "/report_8BAs0TZXB8Ws9zn7KBV93ppcUo5CuywO_FzNxSfxXNs/recorded_audio.wav",
    "/report_9OugJTY_WxUoOJ33vQwAYsol4zHP_dcjAURHmHGKPfA/recorded_audio.wav",
    "/report_QxjI35WATxkcOthJzYrmpFFdVz7Etu70eKJ-WuYocSk/recorded_audio.wav",
    "/report_cWaGlBdGxIHIyIPD1Ac8iJ4fc9tRT1tVEfYKQwbo0ys/recorded_audio.wav",
    "/report_k1S-S-2HekY8y8o-_hNrQvTmYvezomCCJyD7S5_Qn2g/recorded_audio.wav",
    "/report_ohuSPicvcj6wDc064GnEXhdkQW4Z9vJCTq5ItHnkyqM/recorded_audio.wav",
    "/report_mammographie_140/recorded_audio.wav",
    "/report_mammographie_148/recorded_audio.wav",
    "/report_yP4Wtno91JVex6cd4izFsQ06267flev6YdydB8rjebI/recorded_audio.wav",
    "/report_rst1Vqx2nrf-r0U51UuevbXdLwxDnhKnjmYjXJAk1ww/recorded_audio.wav",
    "/report_WPaxdNrHQGE14sctuPQ4DOFp_wb8vtgP5StxXZJ37Hg/recorded_audio.wav",
    
    # === - AUDIOS SUPPLÉMENTAIRES (STRATÉGIQUEMENT SÉLECTIONNÉS) ===
    # Rachis cervical - enregistrements réels variés
    "/report_-O1r169VFVEEU1dE_V7k796-gHSrejx7vP31B3xjwGQ/recorded_audio.wav",
    "/report_25hGADGBJ-9JShEtUkve_-tQuPM5YXMrpvIbGUUeMnw/recorded_audio.wav",
    
    # Rachis lombaire - enregistrements réels variés
    "/report_-Vp1H5PA4aRWlyUun4-_BF7ulOIaQts33AyLx4u99ZY/recorded_audio.wav",
    "/report_CLhHrHn6gUnH_Y85jwkBxTdLV8l988kP4t8dqX88EOM/recorded_audio.wav",
    "/report_5l4qa9ZXxUjtJhjX6aJlYVX4Uyk84V6OxXk4OcE-eBw/recorded_audio.wav",
    "/report_Aw83VqsjTaJPqmJs3V8K62TKdPGf2teimaAraLbUYvk/recorded_audio.wav",
    "/report_LTGcgYQHn3jodezHEcaL4U875_D9lw5uyVj80wJDS-Q/recorded_audio.wav",
    "/report__Y3tcI0v4TGqvFnJi3RR6EIgXZAAqPV_2okj-O_tHpA/recorded_audio.wav",
    "/report_J40fR_huk0Q3eF9dY0CEmu9L4Arf5Brg7zZ_6wZvd4E/recorded_audio.wav",
    
    # Épaule - enregistrements réels variés
    "/report_AvbwWOhHa-4Sc3507QwvwCEl402LfskBhScNvStIwbg/recorded_audio.wav",
    "/report_Frb_V-7-kZmFpmEz39AT8zAkdBEjFEdSciY8SE6_e7w/recorded_audio.wav",
    "/report_znO7zb_1ESS_RL7NlkmnkdylTIvg5SSzSjaDTzZPzKw/recorded_audio.wav",
    "/report_FL6K0Brxi0Ml0GGNplbgRA80x68RAhMtszHSbEMf-lQ/recorded_audio.wav",
    "/report_V67AdLVnpX8i9WEsE6BY4-InmLGWhDqTeYIFQx_eZM4/recorded_audio.wav",
    "/report_U3fjfSYQ5YuxfbU_utlKPy0EjiA1AjBEDI2itIRyFyU/recorded_audio.wav",
    "/report_fdmMA3E8aDoDr0RTFTxNu6RlH9SrwNEyif3Wix0flK0/recorded_audio.wav",
    
    # Thorax/Côtes - enregistrements réels
    "/report_58h1nB0MUCet2VQogEjD6__b9R-PXPedeQPEI9GsEdo/recorded_audio.wav",
    "/report_dazq4xPq_LhPnz2bgyQmYHcKrXx3yNo5auG2swnSmpU/recorded_audio.wav",
    "/report_KqDWw6GF2UHml2c59jAD5n3HLL30iKBAE3j_4f0uTWg/recorded_audio.wav",
    "/report_LxbmrrVRwIASQa_lfBOosw7VZ3SCh6LjtDErc2fOcPs/recorded_audio.wav",
    "/report_s9CD2920VYRGv7duFpImuQ-nxavisGmWGVo0gQ9zfe4/recorded_audio.wav",
    
    # Fractures diverses - enregistrements réels
    "/report_kR-6UhzsPBjFscqHQP_B34dPsbqtd0jI6ZINZA8Ien4/recorded_audio.wav",
    "/report_CnGIjpEu9WIl3lJXeIdIY7Cd-txSUNoMToy_TWma32Y/recorded_audio.wav",
    "/report_Hciy9mOCW3c9hQCzshDT5pxl5ZgeZ87adV6VpAm8TPc/recorded_audio.wav",
    
    # Mammographie - enregistrements réels
    "/report_mammographie_121/recorded_audio.wav",
    "/report_mammographie_138/recorded_audio.wav",
    "/report_mammographie_155/recorded_audio.wav",
    
    # TDM/Examens complexes
    "/report_uU9gmaeXw_oVVi7Bwo0eJ2Cu_pppkKU-FQ1nc38p470/recorded_audio.wav",
    
    # Autres examens variés
    "/report__gqmZnO9sp0ZzFWzH371jWQBL_R6xD91E-Lx1IfJ7-Q/recorded_audio.wav",
    "/report_qZpQYdL_fpYFbUfYbZoh9sR8xjDxgoxwM2XuLNfyl8g/recorded_audio.wav",
    
    # === AUDIOS SYNTHÉTIQUES (pour diversité des voix) ===
    # Voix synthétique 4SFJvuIUvxaPLgk8FoK3 (SPEAKER_23)
    "/report_0Jn2RPfGj_STL7GdrNX7W1-C4-p3rPUjgqZqDD5HgF0/synth_64_4SFJvuIUvxaPLgk8FoK3.mp3",
    "/report_43OifUVMfXwbq397L6I5RIU5knG-m4o0ifC1d4YWtSw/synth_62_4SFJvuIUvxaPLgk8FoK3.mp3",
    "/report_1urL6qQEO-EV2WdsWQ96t8bdd2m8_o8YjN_a0FScaQQ/synth_59_4SFJvuIUvxaPLgk8FoK3.mp3",
    "/report_883DnaoG7JMvagtWymk-_V2YtKJ7Yy5ADA0aviaQB_8/synth_105_4SFJvuIUvxaPLgk8FoK3.mp3",
    "/report_Y8bKZcL78ICsat9Br20mA7p69QQNGFaHgfIY8_N6MVI/synth_106_4SFJvuIUvxaPLgk8FoK3.mp3",
    
    # Voix synthétique mQS95w8LbLFsF6QihxDH (SPEAKER_32)
    "/report_-Vp1H5PA4aRWlyUun4-_BF7ulOIaQts33AyLx4u99ZY/synth_87_mQS95w8LbLFsF6QihxDH.mp3",
    "/report_25hGADGBJ-9JShEtUkve_-tQuPM5YXMrpvIbGUUeMnw/synth_78_mQS95w8LbLFsF6QihxDH.mp3",
    "/report_43OifUVMfXwbq397L6I5RIU5knG-m4o0ifC1d4YWtSw/synth_62_mQS95w8LbLFsF6QihxDH.mp3",
    
    # Voix synthétique FgHDn7bpgpKqz7QttoyC (SPEAKER_36)
    "/report_2TnSIhg-drDPZ2eu54sk6Rj36XSSUxm_NqpZKI7BYyg/synth_145_FgHDn7bpgpKqz7QttoyC.mp3",
    "/report_CfcGA_MYB8L1r5SsyerahlNPN72EUbe6HFy-hQzTK5g/synth_121_FgHDn7bpgpKqz7QttoyC.mp3",
    "/report_GkxZ8fczykCkgCCDNkQ3iSXA12lHZmzfBzhagp_OdVg/synth_117_FgHDn7bpgpKqz7QttoyC.mp3",
    
    # Voix synthétique 6aRkp7Pz4MBOSpUyJCTO (SPEAKER_31)
    "/report_0Jn2RPfGj_STL7GdrNX7W1-C4-p3rPUjgqZqDD5HgF0/synth_64_6aRkp7Pz4MBOSpUyJCTO.mp3",
    "/report_4epGcPG2Wva5Ez2FYaY5qTMVEgwRJCfo6cmHsphqj6s/synth_48_6aRkp7Pz4MBOSpUyJCTO.mp3",
    "/report_6rHb7QWe7u0CS_hHNAPaL8QTX7O0WQXbw7q-IF1-vMg/synth_60_6aRkp7Pz4MBOSpUyJCTO.mp3",
    "/report_7Gwzxi03TtV5kb9Je22o4A-mrMEiMEJQEWvXLxg2ur4/synth_42_6aRkp7Pz4MBOSpUyJCTO.mp3",
    
    # Voix synthétique YwEKgPNrswweXodAZi29 (SPEAKER_37)
    "/report_0q_8Gc9auXyxeX6KC7EoX5KJx3z0vJuvCQYVN2s4JTQ/synth_23_YwEKgPNrswweXodAZi29.mp3",
    "/report_4epGcPG2Wva5Ez2FYaY5qTMVEgwRJCfo6cmHsphqj6s/synth_48_YwEKgPNrswweXodAZi29.mp3",
    "/report_5uAjLRWBJhizfGPRPlgbkZsRIH5K-CF1OO0mWl1yEmk/synth_29_YwEKgPNrswweXodAZi29.mp3",
    "/report_CLhHrHn6gUnH_Y85jwkBxTdLV8l988kP4t8dqX88EOM/synth_31_YwEKgPNrswweXodAZi29.mp3",
    
    # Voix synthétique Ix5oBMHpatfp3naMIpLk (SPEAKER_33)
    "/report_2TnSIhg-drDPZ2eu54sk6Rj36XSSUxm_NqpZKI7BYyg/synth_145_Ix5oBMHpatfp3naMIpLk.mp3",
    "/report_5l4qa9ZXxUjtJhjX6aJlYVX4Uyk84V6OxXk4OcE-eBw/synth_143_Ix5oBMHpatfp3naMIpLk.mp3",
    "/report_TNy7Z2CZXOi_yptA530ZJplGEARTl1yGktATxOvqVFM/synth_111_Ix5oBMHpatfp3naMIpLk.mp3",
    
    # Voix synthétique NZ8KtusXpnktPYja5Qko (SPEAKER_38)
    "/report_0q_8Gc9auXyxeX6KC7EoX5KJx3z0vJuvCQYVN2s4JTQ/synth_23_NZ8KtusXpnktPYja5Qko.mp3",
    "/report_58h1nB0MUCet2VQogEjD6__b9R-PXPedeQPEI9GsEdo/synth_36_NZ8KtusXpnktPYja5Qko.mp3",
    "/report_DtJ3ND4F6oWaiJypWwXgGjVYsW9yIHu9cuA5Pe5ewKM/synth_22_NZ8KtusXpnktPYja5Qko.mp3",
    
    # Voix synthétique K9pQ2PZvpZ94bZfl25YD (SPEAKER_05)
    "/report_FL6K0Brxi0Ml0GGNplbgRA80x68RAhMtszHSbEMf-lQ/synth_10_K9pQ2PZvpZ94bZfl25YD.mp3",
    "/report_M2wTsvq9juqnynWwUSQ3kUhwVruOZzey5faZUfAyuG4/synth_14_K9pQ2PZvpZ94bZfl25YD.mp3",
    "/report_hVxc0Jn2egBrXkz4VN0KXrghVtq0w5VGiCxVWsM2AuI/synth_13_K9pQ2PZvpZ94bZfl25YD.mp3"
]

# Vérification : 100 audios exactement
print(f"Nombre total d'audios sélectionnés : {len(eval_audios)}")
print(f"Audios uniques : {len(set(eval_audios))}")


In [ ]:
fold_path = "/kaggle/input/fr-audio-dataset/Collecte_Audio_FR_Radiologie-20260119T004052Z-3-001/Collecte_Audio_FR_Radiologie"
eval_audios = [fold_path + r_path for r_path in eval_audios]

In [ ]:
# Create a set of audio paths for the test set
test_audio_paths = set(eval_audios)

# Create a new list for test records
test_records = [record for record in records if record["audio"] in test_audio_paths]

# Filter the original records list to remove the test records
records = [record for record in records if record["audio"] not in test_audio_paths]

### Création du Dataset

In [ ]:
# Créer le dataset à partir de la liste
ds = Dataset.from_list(records)
test_ds = Dataset.from_list(test_records)
# Indiquer que la colonne "audio" contient des fichiers audio
#ds = ds.cast_column("audio", Audio(sampling_rate=16_000))

In [ ]:
ds

In [ ]:
test_ds

In [ ]:
def calculate_total_duration(dataset):
    total_duration = 0
    for item in dataset:
        try:
            # torchaudio.info returns metadata about the audio file, including duration
            metadata = torchaudio.info(item['audio'])
            total_duration += metadata.num_frames / metadata.sample_rate
        except Exception as e:
            print(f"Could not process audio file {item['audio']}: {e}")
    return total_duration

# Calculate the total duration for the training and test datasets
total_train_duration = calculate_total_duration(ds)
total_test_duration = calculate_total_duration(test_ds)

print(f"Total duration of training audio files: {total_train_duration / 60:.2f} minutes")
print(f"Total duration of test audio files: {total_test_duration / 60:.2f} minutes")

### Speech Processing

In [ ]:
# !nvidia-smi

In [ ]:
model_name =  "openai/whisper-small"#"leduckhai/MultiMed-ST/asr/whisper-small-french" #"bofenghuang/whisper-small-cv11-french"

In [ ]:
# repo_id = "leduckhai/MultiMed-ST"
# subfolder_path = "asr/whisper-small-french"

In [ ]:
# !pip install --upgrade --force-reinstall --no-cache-dir --no-deps unsloth unsloth_zoo transformers timm

In [ ]:
import torch
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    BitsAndBytesConfig
)

# # 1) Configurer la quantization 4-bits avec bitsandbytes
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit             = True,
#     bnb_4bit_quant_type      = "nf4",
#     bnb_4bit_compute_dtype   = torch.float16
# )

# 2) Charger le Processor (tokenizer + feature_extractor)
processor = WhisperProcessor.from_pretrained(
    "openai/whisper-small",
    use_auth_token = HF_TOKEN,
    language      = "fr",
    task          = "transcribe"
)

# # 3) Charger le modèle en 4-bits sur tous les GPUs dispos
# model = WhisperForConditionalGeneration.from_pretrained(
#     model_name,
#     #quantization_config = bnb_config,   # lien avec bitsandbytes
#     device_map          = "auto",       # répartit sur GPU/CPU
#     trust_remote_code   = True,         # si ton repo contient du code custom
#     token      = HF_TOKEN
# )

# # 4) (optionnel) passer en eval + générer
# model.eval()
# input_vals = processor(audio_array, return_tensors="pt").input_features.to(model.device)
# generated_ids = model.generate(input_vals)
# transcript    = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
# print(transcript)


In [ ]:
def model_summary(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    percent = 100 * trainable_params / total_params

    print(f"{'Type':<20} {'Count':>15}")
    print("-" * 35)
    print(f"{'Total parameters':<20} {total_params:>15,}")
    print(f"{'Trainable parameters':<20} {trainable_params:>15,}")
    print(f"{'Trainable %':<20} {percent:>14.2f} %")

In [ ]:
from huggingface_hub import snapshot_download
from adapters import WhisperAdapterModel, LoRAConfig
from torch.optim import AdamW

# local_model_path = snapshot_download(
#     repo_id="leduckhai/MultiMed-ST",
#     allow_patterns="asr/whisper-small-french/*",
#     local_dir="./multimed_local"
# )
# model_path = "./multimed_local/asr/whisper-small-french/checkpoint"

model = WhisperAdapterModel.from_pretrained(
    model_name,
    # model_path,
    # local_files_only=True,
    torch_dtype=torch.float32 if torch.cuda.is_available() else None,
    device_map="auto",   # ou "auto" si tu veux
)
        

lora_cfg = LoRAConfig(
    r=16, #32
    alpha=32, #64 
    dropout=0.1, # à revoir
    # Emplacements LoRA (attention + FFN)
    selfattn_lora=True,          # injecter dans l’attention
    #crossattn_lora=True,   
    attn_matrices=["q", "k", "v", "o"],  # équiv. q_proj, k_proj, v_proj, o_proj(depend de QKV donc pas nécessaire)
    intermediate_lora=True,      # FFN (W_up/W_down) si tu veux l’équiv. up/down_proj
    output_lora=True,            # couche de sortie du bloc/ out du MLP (pas résidus ni NormLayer)
)

# # Geler l'encodeur
for param in model.model.encoder.parameters():
       param.requires_grad = False

# Dégeler les 4 premières couches
for i in range(4):
    prefix = f"model.encoder.layers.{i}"   # adapter si nécessaire: "model.model.encoder.layers.{i}"
    for name, p in model.named_parameters():
        if name.startswith(prefix):
            p.requires_grad = True

# 3) Ajouter/activer l’adapter
adapter_name = "joint_rad_africa_adapter"#"encoder_4_layer_decoder" #"decoder_lora"
model.add_adapter(adapter_name, config=lora_cfg, set_active=True)

print("Put only Encoder_first_4_layers & Decoder trainable")
print("                   ⇩⇩⇩⇩⇩                ")
model_summary(model)
print("\nAfter LoRA Filtering")
print("                   ⇩⇩⇩⇩⇩                ")
print(model.adapter_summary())

In [ ]:
def print_trainable_parameters_summary(model):
    trainable_params = 0
    all_param = 0
    
    # Dictionnaires pour le détail
    summary = {"Encoder": 0, "Decoder": 0, "Others": 0}
    
    for name, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
            if "encoder" in name:
                summary["Encoder"] += param.numel()
            elif "decoder" in name:
                summary["Decoder"] += param.numel()
            else:
                summary["Others"] += param.numel()

    print("\n" + "="*50)
    print(f"📊 RÉSUMÉ DES PARAMÈTRES ENTRAÎNABLES (LoRA)")
    print("="*50)
    print(f"🔹 TOTAL ENTRAÎNABLE : {trainable_params:,} ({100 * trainable_params / all_param:.4f}%)")
    print("-" * 50)
    print(f"  ∟ 👂 ENCODEUR (Accent) : {summary['Encoder']:,} params")
    print(f"  ∟ 🧠 DÉCODEUR (Jargon) : {summary['Decoder']:,} params")
    print(f"  ∟ ⚙️  AUTRES            : {summary['Others']:,} params")
    print("="*50 + "\n")

# Appel de la fonction
print_trainable_parameters_summary(model)

In [ ]:
import bitsandbytes as bnb

params_encoder_4 = []
params_decoder = []

for name, param in model.named_parameters():
    if param.requires_grad:
        if "encoder" in name:
            params_encoder_4.append(param)
        elif "decoder" in name:
            params_decoder.append(param)

# Stratégie de Joint Training
optimizer_grouped_parameters = [
    {
        "params": params_encoder_4,
        "lr": 8e-6,  # LR très bas : on ne veut pas casser l'alignement phonétique
        "weight_decay": 0.01
    },
    {
        "params": params_decoder,
        "lr": 4e-5,  # LR plus haut : le décodeur doit assimiler le jargon radiologique
        "weight_decay": 0.01 # weight_decay (décroissance des poids) autour de 0.01 ou 0.05. Cela agira comme une régularisation supplémentaire pour éviter que vos adaptateurs LoRA ne mémorisent vos audios par cœur (overfitting).
    }
]

optimizer = bnb.optim.AdamW8bit(optimizer_grouped_parameters)
# optimizer = AdamW(optimizer_grouped_parameters)

In [ ]:
target_dtype = torch.float32  # si tu es en fp16
model.to(dtype=target_dtype)  # FAIRE APRES avoir attaché les adapters/LoRA

#### Data Prep

#### Segmentation des audios longs (>30s)

**Problème** : Whisper a une contrainte architecturale de 30 secondes (3000 frames mel).
Nos audios font ~2 minutes → mismatch catastrophique audio/label.

**Solution** : Segmentation avec chevauchement + répartition proportionnelle du texte.

In [ ]:
# import torchaudio
# import torch
# import numpy as np
# from tqdm import tqdm

# def segment_audio_with_text(audio_path, text, segment_duration=25, overlap=5, sr=16000):
#     """
#     Segmente un audio long ET répartit le texte proportionnellement.
    
#     Args:
#         audio_path: chemin vers le fichier audio
#         text: transcription complète
#         segment_duration: durée de chaque segment en secondes (< 30s pour sécurité)
#         overlap: chevauchement en secondes pour continuité contextuelle
#         sr: sampling rate cible
    
#     Returns:
#         Liste de dicts {audio_array, text_segment, duration}
#     """
#     try:
#         waveform, orig_sr = torchaudio.load(audio_path)
#         if orig_sr != sr:
#             waveform = torchaudio.functional.resample(waveform, orig_sr, sr)
        
#         waveform = waveform.squeeze()  # [T]
#         total_samples = waveform.shape[0]
#         total_duration = total_samples / sr
        
#         # Si audio <= 30s, pas de segmentation nécessaire
#         if total_duration <= 30:
#             return [{
#                 "audio_array": waveform.numpy(),
#                 "text_segment": text,
#                 "duration": total_duration
#             }]
        
#         # Calcul des segments
#         segment_samples = int(segment_duration * sr)
#         hop_samples = int((segment_duration - overlap) * sr)
        
#         segments = []
#         words = text.split()
#         total_words = len(words)
        
#         # Estimation : répartition proportionnelle des mots par durée
#         words_per_second = total_words / total_duration if total_duration > 0 else 0
        
#         start = 0
#         while start < total_samples:
#             end = min(start + segment_samples, total_samples)
#             audio_seg = waveform[start:end].numpy()
            
#             # Estimation des mots pour ce segment temporel
#             seg_start_time = start / sr
#             seg_end_time = end / sr
#             word_start = int(seg_start_time * words_per_second)
#             word_end = int(seg_end_time * words_per_second)
            
#             # Pour le dernier segment, prendre tous les mots restants
#             if end >= total_samples:
#                 word_end = total_words
            
#             text_seg = " ".join(words[word_start:word_end])
            
#             # Vérifier que le segment est valide
#             seg_duration = len(audio_seg) / sr
#             if seg_duration >= 1.0 and len(text_seg.strip()) > 5:
#                 segments.append({
#                     "audio_array": audio_seg,
#                     "text_segment": text_seg,
#                     "duration": seg_duration
#                 })
            
#             start += hop_samples
        
#         return segments
        
#     except Exception as e:
#         print(f"[ERREUR SEGMENTATION] {audio_path}: {e}")
#         return []


# def create_segmented_dataset(records, processor, segment_duration=25, overlap=5, max_label_tokens=440):
#     """
#     Crée un dataset segmenté prêt pour l'entraînement Whisper.
#     Retourne input_features, attention_mask et labels pour chaque sample.
    
#     Args:
#         records: liste de dicts {"audio": path, "text": transcription}
#         processor: WhisperProcessor
#         segment_duration: durée segment en secondes
#         overlap: chevauchement en secondes
#         max_label_tokens: limite de tokens pour les labels (< 448 pour sécurité)
#     """
#     all_samples = []
#     stats = {"total_segments": 0, "skipped_audio": 0, "skipped_label": 0, "original_count": len(records)}
    
#     for record in tqdm(records, desc="🔄 Segmentation des audios"):
#         segments = segment_audio_with_text(
#             record["audio"], 
#             record["text"],
#             segment_duration=segment_duration,
#             overlap=overlap
#         )
        
#         for seg in segments:
#             try:
#                 # Feature extraction (mel spectrogram)
#                 inputs = processor.feature_extractor(
#                     seg["audio_array"], 
#                     sampling_rate=16000, 
#                     return_tensors="pt"
#                 )
                
#                 # Vérifier que les features ne sont pas trop courtes
#                 if inputs.input_features.shape[-1] < 100:  # < ~1 seconde
#                     stats["skipped_audio"] += 1
#                     continue
                
#                 # === ATTENTION MASK (comme prepare_batch) ===
#                 # Créer le masque d'attention basé sur les positions non-nulles
#                 # Pour Whisper: shape [80, T] -> mask [T]
#                 input_feats = inputs.input_features[0]  # [80, T]
                
#                 # Méthode 1: Masque basé sur l'énergie (positions où le signal existe)
#                 # On considère qu'une frame est "active" si elle a de l'énergie
#                 energy = input_feats.abs().sum(dim=0)  # Somme sur les 80 bins mel -> [T]
#                 attention_mask = (energy > 0).long()  # 1 si énergie > 0, sinon 0
                
#                 # Alternative plus simple: tout à 1 car on a du vrai audio (pas de padding encore)
#                 # attention_mask = torch.ones(input_feats.shape[-1], dtype=torch.long)
                
#                 # Tokenization du label
#                 labels = processor.tokenizer(
#                     seg["text_segment"],
#                     truncation=True,
#                     max_length=max_label_tokens,
#                     return_tensors="pt"
#                 ).input_ids
                
#                 # Vérifier que le label n'est pas trop court
#                 if labels.shape[-1] < 5:
#                     stats["skipped_label"] += 1
#                     continue
                
#                 # === RETOURNER LES 3 ÉLÉMENTS ===
#                 all_samples.append({
#                     "input_features": input_feats,           # [80, T]
#                     "attention_mask": attention_mask,         # [T]
#                     "labels": labels[0]                       # [L]
#                 })
#                 stats["total_segments"] += 1
                
#             except Exception as e:
#                 print(f"[ERREUR TRAITEMENT] {e}")
#                 stats["skipped_audio"] += 1
    
#     # Rapport
#     print("\n" + "="*60)
#     print("📊 RAPPORT DE SEGMENTATION")
#     print("="*60)
#     print(f"  📁 Fichiers originaux    : {stats['original_count']}")
#     print(f"  ✅ Segments créés        : {stats['total_segments']}")
#     print(f"  ⚠️  Segments audio ignorés: {stats['skipped_audio']}")
#     print(f"  ⚠️  Segments label ignorés: {stats['skipped_label']}")
#     print(f"  📈 Ratio expansion       : {stats['total_segments']/stats['original_count']:.2f}x")
#     print("="*60)
    
#     return Dataset.from_list(all_samples)


# # Analyse des durées pour choisir les bons paramètres
# print("📏 Analyse des durées audio...")
# durations = []
# for record in tqdm(records[:50], desc="Échantillonnage"):  # Échantillon de 50
#     try:
#         info = torchaudio.info(record["audio"])
#         durations.append(info.num_frames / info.sample_rate)
#     except:
#         pass

# if durations:
#     print(f"\n📊 Statistiques durées (échantillon de {len(durations)} audios):")
#     print(f"   Min: {min(durations):.1f}s | Max: {max(durations):.1f}s | Moyenne: {np.mean(durations):.1f}s")
#     print(f"   Audios > 30s: {sum(1 for d in durations if d > 30)} ({100*sum(1 for d in durations if d > 30)/len(durations):.1f}%)")

In [ ]:
# # ============================================
# # 🚀 CRÉATION DU DATASET SEGMENTÉ
# # ============================================
# # Paramètres optimaux pour tes audios de ~2min:
# #   - segment_duration=25s : laisse 5s de marge vs limite 30s
# #   - overlap=5s : continuité contextuelle entre segments
# #   - max_label_tokens=440 : marge de sécurité vs limite 448

# ds_segmented = create_segmented_dataset(
#     records, 
#     processor,
#     segment_duration=25,  # secondes par segment
#     overlap=5,            # chevauchement en secondes
#     max_label_tokens=440  # limite tokens label
# )

# print(f"\n✅ Dataset segmenté créé: {len(ds_segmented)} échantillons")

In [ ]:
# # ============================================
# # 📊 SPLIT TRAIN/TEST SUR LE DATASET SEGMENTÉ
# # ============================================
# from datasets import DatasetDict

# # Split: 80% train, 20% validation
# split_segmented = ds_segmented.train_test_split(test_size=0.2, seed=42)

# ds_prepared = DatasetDict({
#     "train": split_segmented["train"],
#     "test": split_segmented["test"]
# })

# print(f"📊 Dataset final:")
# print(f"   Train: {len(ds_prepared['train'])} segments")
# print(f"   Test:  {len(ds_prepared['test'])} segments")

# # Vérification d'un échantillon
# sample = ds_prepared["train"][0]
# print(f"\n🔍 Vérification échantillon:")
# # Dataset convertit les tensors en listes, donc on utilise np.array pour avoir la shape
# input_feats = np.array(sample['input_features'])
# attn_mask = np.array(sample['attention_mask'])
# labels = sample['labels']
# print(f"   input_features shape: {input_feats.shape}")  # Devrait être (80, T)
# print(f"   attention_mask shape: {attn_mask.shape}")    # Devrait être (T,)
# print(f"   labels length: {len(labels)}")

In [ ]:
from tqdm import tqdm

In [ ]:
import soundfile as sf
import torchaudio
import torch
from tqdm import tqdm
from datasets import Dataset

def load_audio_robust(audio_path):
    """Charge l'audio avec fallback torchaudio si soundfile échoue."""
    try:
        # Essayer soundfile d'abord (plus rapide)
        audio, sr = sf.read(audio_path)
        audio = torch.from_numpy(audio).float()
    except Exception:
        # Fallback sur torchaudio pour formats non standards
        audio, sr = torchaudio.load(audio_path)
        audio = audio.squeeze()
    
    # Convertir stéréo → mono si nécessaire
    if audio.dim() > 1:
        audio = audio.mean(dim=-1)
    
    # Resample si nécessaire
    if sr != 16000:
        audio = torchaudio.functional.resample(audio, sr, 16000)
    
    return audio.numpy()

def prepare_single_sample(sample):
    """Prépare un échantillon audio."""
    audio = load_audio_robust(sample["audio"])
    
    inputs = processor.feature_extractor(
        audio, 
        sampling_rate=16000, 
        return_tensors="pt"
    )
    
    labels = processor.tokenizer(
        sample["text"], 
        truncation=True, 
        max_length=448, 
        return_tensors="pt"
    ).input_ids
    
    return {
        "input_features": inputs.input_features[0],
        "labels": labels[0]
    }

# Traitement avec boucle manuelle
print("🔄 Traitement des audios...")
processed_data = []
skipped_count = 0

for i in tqdm(range(len(ds)), desc="Processing audios"):
    try:
        result = prepare_single_sample(ds[i])
        processed_data.append(result)
    except Exception as e:
        skipped_count += 1
        if skipped_count <= 5:  # Afficher seulement les 5 premières erreurs
            print(f"\n[SKIP {skipped_count}] Index {i}: {str(e)[:100]}")
        continue

# Créer le dataset
ds_prepared = Dataset.from_list(processed_data)
print(f"✅ Dataset créé: {len(ds_prepared)} échantillons")
print(f"⚠️  Fichiers ignorés: {skipped_count}")

In [ ]:
from datasets import DatasetDict

# Split: 80% train, 20% test
split_dataset = ds_prepared.train_test_split(test_size=0.2, seed=42)

# Organiser sous forme de DatasetDict
ds_prepared = DatasetDict({
    "train": split_dataset["train"],
    "test": split_dataset["test"]
})

In [ ]:
ds_prepared

In [ ]:
import matplotlib.pyplot as plt
import torch
import numpy as np

features = ds_prepared['train'][91]["input_features"]
plt.figure(figsize=(10, 4))
plt.imshow(np.array(features), aspect="auto", origin="lower", interpolation="none")
plt.title("Spectrogramme log-Mel (Whisper)")
plt.xlabel("Time")
plt.ylabel("Mel bins")
plt.colorbar()
plt.show()

### Fine Tuning

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
# from unsloth import is_bf16_supported

In [ ]:
#Set this to the language you want to train on
model.generation_config.language = "<|fr|>"
model.generation_config.task = "transcribe"
model.config.suppress_tokens = []
model.generation_config.forced_decoder_ids = None

In [ ]:
# model.generation_config.language = "fr"
# model.generation_config.task = "transcribe"

# model.generation_config.forced_decoder_ids = [[1, None], [2, 50359]] #None

In [ ]:
"""
# decoder_start_token_id
C’est un token spécial que le modèle doit recevoir au tout début de la séquence cible pendant l'entraînement.
Sans ce token de démarrage, le modèle ne saura pas où commencer la génération lors du décodage.

Il faut un ajout explicite du decoder_start_token_id au début de la séquence cible.
"""

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    """
    Data Collator optimisé pour la segmentation Whisper.
    Gère le padding/truncation à exactement 3000 frames mel (30s).
    Utilise les attention_mask pré-calculés dans create_segmented_dataset.
    """
    def __init__(self, processor, target_dtype=torch.float32, max_label_len=448):
        self.processor = processor
        self.target_dtype = target_dtype
        self.max_label_len = max_label_len
        self.max_audio_frames = 3000  # 30 secondes = 3000 frames mel à 100Hz

    def __call__(self, features):
        # === AUDIO (encoder) + ATTENTION MASK ===
        processed_features = []
        processed_masks = []
        
        for f in features:
            feat = f["input_features"]
            mask = f.get("attention_mask", None)
            
            # Gérer le cas où feat est un numpy array
            if not isinstance(feat, torch.Tensor):
                feat = torch.tensor(feat)
            if mask is not None and not isinstance(mask, torch.Tensor):
                mask = torch.tensor(mask)
            
            T_original = feat.shape[-1]
            
            # Truncation si > 3000 frames
            if T_original > self.max_audio_frames:
                feat = feat[..., :self.max_audio_frames]
                if mask is not None:
                    mask = mask[:self.max_audio_frames]
            # Padding si < 3000 frames
            elif T_original < self.max_audio_frames:
                pad_len = self.max_audio_frames - T_original
                feat = torch.nn.functional.pad(feat, (0, pad_len), value=0)
                if mask is not None:
                    # Padding du masque avec des 0 (positions à ignorer)
                    mask = torch.nn.functional.pad(mask, (0, pad_len), value=0)
            
            # Si pas de masque fourni, créer un masque basique
            if mask is None:
                mask = torch.ones(self.max_audio_frames, dtype=torch.long)
                # Mettre à 0 les positions paddées
                if T_original < self.max_audio_frames:
                    mask[T_original:] = 0
            
            processed_features.append({"input_features": feat})
            processed_masks.append(mask)
        
        # Batch des features audio
        batch = self.processor.feature_extractor.pad(
            processed_features, return_tensors="pt"
        )
        batch["input_features"] = batch["input_features"].to(self.target_dtype)
        
        # Stack des masques d'attention encodeur
        batch["attention_mask"] = torch.stack(processed_masks, dim=0).long()

        # === LABELS (decoder) ===
        label_features = []
        for f in features:
            lab = f["labels"]
            if not isinstance(lab, torch.Tensor):
                lab = torch.tensor(lab)
            # Truncation des labels si nécessaire
            label_features.append({"input_ids": lab[:self.max_label_len]})
        
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"]
        dec_attn = labels_batch["attention_mask"]

        # Ignorer la loss sur les tokens PAD (-100)
        labels = labels.masked_fill(dec_attn.ne(1), -100)

        batch["labels"] = labels
        batch["decoder_attention_mask"] = dec_attn
        
        return batch

In [ ]:
target_dtype = next(model.parameters()).dtype   # ex: torch.float16 si fp16
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor, target_dtype=target_dtype, max_label_len=448
)

In [ ]:
# data_collator = DataCollatorSpeechSeq2SeqWithPadding(
#     processor=processor,
#     decoder_start_token_id=model.config.decoder_start_token_id,
# )

Evaluation

In [ ]:
import evaluate

metric = evaluate.load("wer")

In [ ]:
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}

2.0:
le but principal est l’adaptation acoustique (accent/bruit/mic/locuteurs).
Geler le décodeur et n’entraîner qu’un petit morceau de l’encodeur est souvent le meilleur move pour ça. Mais il faut le faire progressivement et avec très peu de degrés de liberté, sinon tu retombes dans l’oubli catastrophique.

In [ ]:
from transformers import Seq2SeqTrainer
import torch

class WhisperCleanGenTrainer(Seq2SeqTrainer):
    def prediction_step(
        self,
        model,
        inputs,
        prediction_loss_only=False,
        ignore_keys=None,
        **gen_kwargs,
    ):
        # Déplacement sur device / dtypes géré par le Trainer
        inputs = self._prepare_inputs(inputs)

        # 1) Calcul du loss (avec labels) -- pas de génération ici
        loss = None
        if "labels" in inputs:
            with torch.no_grad():
                with self.compute_loss_context_manager():
                    loss = self.compute_loss(model, inputs, return_outputs=False)

        # 2) Build d'entrées propres pour generate (PAS de labels/decoder_input_ids/etc.)
        gen_inputs = {
            k: v
            for k, v in inputs.items()
            if k in ("input_features", "attention_mask", "encoder_outputs")
        }

        generated_tokens = None
        if self.args.predict_with_generate and not prediction_loss_only:
            gen_kwargs = getattr(self, "_gen_kwargs", {}) | gen_kwargs
            with torch.no_grad():
                generated_tokens = model.generate(**gen_inputs, **gen_kwargs)

        labels = inputs.get("labels", None)
        return (loss, generated_tokens, labels)


In [ ]:
import os
import wandb

WANDB_API_KEY = os.getenv("WANDB_API_KEY")
if WANDB_API_KEY:
    wandb.login(key=WANDB_API_KEY)
else:
    print("WANDB_API_KEY not set; wandb login skipped.")

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./Whisper-AfroRad-FR-2",
    seed = 3407,
    predict_with_generate=True,
    per_device_train_batch_size=12, # à revoir
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,  # increase by 2x for every 2x decrease in batch size
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    eval_strategy="steps",
    save_strategy="steps",
    logging_strategy="steps",
    eval_steps=100,
    save_steps=100,
    max_steps=1000,
    generation_max_length=model.config.max_target_positions,  #225,
    learning_rate=3e-5,  #ne sera plus utilisée pour les paramètres du modèle, mais elle peut servir de référence pour le scheduler
    num_train_epochs=3,
    bf16=True,
    fp16=False,
    report_to=["wandb"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=True,
    remove_unused_columns=False,  # required as the PeftModel forward doesn't have the signature of the wrapped model's forward
    label_names=["labels"],  # same reason as above
)


In [ ]:
trainer = WhisperCleanGenTrainer(
    args=training_args,
    model=model,
    train_dataset=ds_prepared["train"],
    eval_dataset=ds_prepared["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.tokenizer,
    optimizers=(optimizer, None) # Injecte l'optimiseur différentiel    
)

# from transformers.integrations import TensorBoardCallback

# trainer.remove_callback(TensorBoardCallback)  # retire la callback qui sérialise le config

trainer.train()

In [ ]:
kwargs = {
    "language": "fr",
    "model_name": "Whisper Small FR - Radiologie", # 2xConv1D + Encoder-Layer[0:3]+ LoRa(QKVO; FFN)
    "finetuned_from": "openai/whisper-small",
    "tasks": "automatic-speech-recognition",
}
trainer.push_to_hub(**kwargs)